# 0. Setup

In [1]:
import ibis
import pandas as pd
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="model")
con = ibis.duckdb.connect(dirs.db_path, read_only=True)

In [2]:
%%script true
from ibis import _, selectors as s
import pandas as pd

# 1. Execute the native Ibis aggregation (returns a single wide row)
t_panel = con.table("working_yearly_with_tfp_wave")
nb_rows = t_panel.count().execute()
raw_result = (
    t_panel
    .select(
        'tfp',
        'peer_tfp_ttwa_donut', 'peer_tfp_pc4_donut', 'peer_tfp_pc8',
        'tfp_wav1', 'tfp_wav2', 'tfp_wav3', 'nb_peers'
    )
    .aggregate(
        s.across(
            s.all(),
            {
                "count": _.count(),
                "percent": _.count() / nb_rows,
                "mean": _.mean(),
                "median": _.median(),
                "min": _.min(),
                "max": _.max(),
                "std": _.std(),
            }
        )
    )
    .execute()
)

# 2. Reshape the single row into a MultiIndex Series, then unstack
s_flat = raw_result.iloc[0]
s_flat.index = pd.MultiIndex.from_tuples(
    [col.rsplit("_", 1) for col in s_flat.index], 
    names=["variable", "metric"]
)

# 3. Pivot the metrics into separate columns
df_count = (
    s_flat
    .unstack(level="metric")[["count", "percent", "mean", "median", "min", "max", "std"]]
)

display(df_count.style.format({
    "count": "{:,.0f}",
    "percent": "{:.1%}",
    "mean": "{:,.3f}",
    "median": "{:,.3f}",
    "min": "{:,.3f}",
    "max": "{:,.3f}",
    "std": "{:,.3f}"
}))

In [3]:
import ibis
from ibis import _
import traceback
import pandas as pd
import numpy as np
from dataclasses import dataclass, field, asdict

import statsmodels.api as sm
from linearmodels.panel import PanelOLS
from linearmodels.panel.results import PanelEffectsResults

@dataclass
class ModelSpec():
    Y: str
    X: list[str]
    W: list[str] = field(default_factory=list)
    to_log: list[str] = field(default_factory=list)
    fe: list[str] = field(default_factory=list)
    description: str = ""
    include: bool = True

def run_panel(args: tuple[ModelSpec, pd.DataFrame, str]) -> tuple[PanelEffectsResults, str | dict[str, float], dict[str, pd.Series | None]]:
    mod, table_indicator, model_name = args
    print(f"Running model '{model_name}'")
    try:

        params_named = [mod.Y] + mod.X + mod.W
        params_named_full = ['registered_number', 'year'] + params_named
        if 'Y' in mod.to_log:
            mod.to_log.append(mod.Y)
        if 'X' in mod.to_log:
            mod.to_log.extend(mod.X)
        if 'W' in mod.to_log:
            mod.to_log.extend(mod.W)
        should_log_Y = mod.Y in mod.to_log
        param_Y = f'ln_{mod.Y}' if should_log_Y else mod.Y
        params_transformed = [f'ln_{p}' if p in mod.to_log else p for p in params_named]
        params_regressors = [p for p in params_transformed if p != param_Y]
        params_full = ['registered_number', 'year'] + params_transformed
        # 1. Execute into a Pandas DataFrame and set the MultiIndex for linearmodels
        if isinstance(table_indicator, str):
            t_panel: ibis.Table = con.table(table_panel_name, read_only=True)
            df_model = (
                t_panel
                .select(['registered_number', 'year'] + params_named)
                .mutate(**{
                    f'ln_{p}': _[p].log() for p in params_full for p in params_named if p in mod.to_log
                })
                .drop_null(params_full)
                .select(params_full)
                .execute()
                .set_index(['registered_number', 'year'])
            )
        elif isinstance(table_indicator, pd.DataFrame):
            df_raw = table_indicator
            df_model = (
                df_raw
                .filter(items=params_named_full)
                .assign(**{
                    f'ln_{p}': df_raw[p].apply(lambda x: np.log(x) if x > 0 else None) for p in params_named if p in mod.to_log
                })
                .dropna(subset=params_full)
                .filter(items=params_full)
                .set_index(['registered_number', 'year'])
            )
        else:
            raise ValueError(f"Invalid table_indicator type: {type(table_indicator)}. Must be str or pd.DataFrame.")
        
        Y = df_model[param_Y]
        X = sm.add_constant(df_model[list(params_regressors)])
        
        # 2. Estimate the model with Firm and Year Fixed Effects
        mod_panel = PanelOLS(Y, X, entity_effects='i' in mod.fe, time_effects='t' in mod.fe)
        res = mod_panel.fit(cov_type='clustered', cluster_entity=True)
        
        # Extract \beta results iterating through full_params
        beta: dict[str, float] = { p: res.params[p] for p in params_regressors }
        effects_dict: dict[str, pd.Series | None] = {
            'i': res.estimated_effects.xs('entity_effects', level=1) if 'entity_effects' in res.estimated_effects.index.names else None,
            't': res.estimated_effects.xs('time_effects', level=1) if 'time_effects' in res.estimated_effects.index.names else None
        }

        print(f"✅ Model '{model_name}' estimated: {", ".join([f'{k}={v:.3f}' for k, v in beta.items()])}")
        return res, beta, effects_dict
    
    except Exception as e:
        print(f"❌ Model '{model_name}' failed. {type(e).__name__}: {e}")
        traceback.print_exc()  # Print the full traceback for debugging
        with open(dirs.output_dir / "error.log", "a") as f:
            f.write(f"Model '{model_name}' failed. {type(e).__name__}: {e}\n")
            traceback.print_exc(file=f)
        return None, None, None     #type: ignore

def format_str(res_serie: tuple[PanelEffectsResults, ModelSpec, str]) -> str:
    res, mod, model_name = res_serie
    
    # 5. Store the results and parameters
    param_str = ""
    param_str += "Model Parameters:\n"
    for key, value in asdict(mod).items():
        if isinstance(value, list) and len(value) == 0:
            continue
        if value is None:
            continue
        param_str += f"{key}: {value}\n"

    output_str = ""
    output_str += f"Model '{model_name}': {mod.description}\n"
    output_str += param_str + "\n"
    output_str += f"{res.summary}\n\n"
    output_str += "=" * 87 + "\n\n"

    return output_str

# 0. 2-factor productivity

In [4]:
table_panel_name = "working_yearly_with_tfp_wave"
out_file_name = "results_0_tfp"
t_panel = con.table(table_panel_name)
df_panel = t_panel.execute()

models = {
    'gva1_ft': {'Y': 'gva1', 'X': ['total_assets', 'employees'], 'to_log': ['Y', 'X'], 'fe': ['i', 't'], 'description': 'GVA = renumeration + EBITDA, K = total assets, firm and time fixed effects'},
    'k2_ft': {'Y': 'gva1', 'X': ['fixed_total', 'employees'], 'to_log': ['Y', 'X'], 'fe': ['i', 't'], 'description': 'GVA = renumeration + EBITDA, K = fixed assets (no current assets), firm and time fixed effects'},
    'gva1_f': {'Y': 'gva1', 'X': ['total_assets', 'employees'], 'to_log': ['Y', 'X'], 'fe': ['i'], 'description': 'GVA = renumeration + EBITDA, K = total assets, firm only fixed effects'},
    'gva1_t': {'Y': 'gva1', 'X': ['total_assets', 'employees'], 'to_log': ['Y', 'X'], 'fe': ['t'], 'description': 'GVA = renumeration + EBITDA, K = total assets, time only fixed effects'},
    'gva1_': {'Y': 'gva1', 'X': ['total_assets', 'employees'], 'to_log': ['Y', 'X'], 'fe': [],('description'): ('GVA = renumeration + EBITDA, K = total assets, no fixed effects')}
}

run_res_series: list[tuple[PanelEffectsResults, ModelSpec, str]] = []
worker_args = [
    (ModelSpec(**mod_obj), df_panel, m_name)
    for m_name, mod_obj in models.items() if ModelSpec(**mod_obj).include
]
for args in worker_args:
    mod, table_panel_name, model_name = args
    res, beta, effects_dict = run_panel(args)
    if res is None:
        print(f"Model '{model_name}' failed. Skipping.")
        continue
    run_res_series.append((res, mod, model_name))

model_count = len(run_res_series)
if model_count:
    output_str = "\n".join(map(format_str, run_res_series))
    print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}, writing.")
    with open(dirs.output_dir / f"{out_file_name}.txt", "w") as f:
        f.write(output_str)

Running model 'gva1_ft'
✅ Model 'gva1_ft' estimated: ln_total_assets=0.265, ln_employees=0.621
Running model 'k2_ft'
✅ Model 'k2_ft' estimated: ln_fixed_total=0.058, ln_employees=0.717
Running model 'gva1_f'
✅ Model 'gva1_f' estimated: ln_total_assets=0.264, ln_employees=0.618
Running model 'gva1_t'
✅ Model 'gva1_t' estimated: ln_total_assets=0.422, ln_employees=0.559
Running model 'gva1_'
✅ Model 'gva1_' estimated: ln_total_assets=0.422, ln_employees=0.559
Panel regressions complete. 5 models, writing.


In [5]:
out_file_name = "results_0_tfp"
if model_count:
    output_str = "\n".join(map(format_str, run_res_series))
    print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}, writing.")
    with open(dirs.output_dir / f"{out_file_name}.txt", "w") as f:
        f.write(output_str)

Panel regressions complete. 5 models, writing.


# 1a. LMM
$$
\begin{align*}
y_{it} &= \alpha_i + \gamma_t + w_{it}\theta + \beta E[TFP_{-i,g,t}\vert{}g] + \epsilon_{it} \\
x_{it} &=
\begin{pmatrix}
k_{it} & l_{it}
\end{pmatrix}
\end{align*}
$$
**Run 1**
- ✅ Model 'peer3' estimated: peer_tfp_ttwa_donut=0.033, peer_tfp_pc4_donut=0.015, peer_tfp_pc8=0.280
- ✅ Model 'peer3_no_firm_fe' estimated: peer_tfp_ttwa_donut=0.204, peer_tfp_pc4_donut=0.142, peer_tfp_pc8=0.559
- ✅ Model 'peer3_employees' estimated: ln_employees=0.009, peer_tfp_ttwa_donut=0.033, peer_tfp_pc4_donut=0.015, peer_tfp_pc8=0.280
Panel regressions complete. 3 models, writing.

In [34]:
from concurrent.futures import ProcessPoolExecutor
import numpy as np

table_panel_name = "working_yearly_with_tfp_wave"
t_panel = con.table(table_panel_name)
df_panel = t_panel.execute()
df_panel_interact = df_panel.copy()
df_panel_interact['peer_tfp_pc8_x_employees'] = df_panel_interact['peer_tfp_ttwa_donut'] * np.log(df_panel_interact['employees'])

# 3. Models: varying Y (gva1 vs gva2) and K (fixed_total vs tangibles)
models = {
    'lmm': {
        'Y': 'tfp',
        'X': ['peer_tfp_ttwa_donut', 'peer_tfp_pc4_donut', 'peer_tfp_pc8'],
        'fe': ['i', 't'],
        'description': 'Base: 3 donut peer TFP effects, firm time fixed effects, no controls'
    },
    'lmm_no_firm_fe': {
        'Y': 'tfp',
        'X': ['peer_tfp_ttwa_donut', 'peer_tfp_pc4_donut', 'peer_tfp_pc8'],
        'fe': ['t'],
        'description': 'Base - firm FE'
    },
    'lmm_employees': {
        'Y': 'tfp',
        'X': ['peer_tfp_ttwa_donut', 'peer_tfp_pc4_donut', 'peer_tfp_pc8'],
        'W': ['employees', 'peer_tfp_pc8_x_employees'],
        'to_log': ['employees'],
        'fe': ['i', 't'],
        'description': 'Base + employees interaction'
    }
}

run_res_series: list[tuple[PanelEffectsResults, ModelSpec, str]] = []
worker_args = [
    (ModelSpec(**mod_obj), df_panel_interact, m_name)
    for m_name, mod_obj in models.items() if ModelSpec(**mod_obj).include
]
# with ProcessPoolExecutor(max_workers=4) as executor:
#     panel_raw = executor.map(run_panel, worker_args)
#     i = 0
#     run_res_series.append(panel_raw) #type: ignore
#     # for panel_out in panel_raw:
#     #     if panel_out[0] is not None:
#     #         continue
#     #     res = panel_out[0]
#     #     mod = worker_args[i][0]
#     #     model_name = worker_args[i][2]
#     #     if res is None:
#     #         print(f"Model '{model_name}' failed. Skipping.")
#     #         continue
#     #     run_res_series.append((res, mod, model_name))
#     #     i += 1
for args in worker_args:
    mod, table_panel_name, model_name = args
    res, beta, effects_dict = run_panel(args)
    if res is None:
        print(f"Model '{model_name}' failed. Skipping.")
        continue
    run_res_series.append((res, mod, model_name))

model_count = len(run_res_series)
if model_count:
    output_str = "\n".join(map(format_str, run_res_series))
    print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}, writing.")
    with open(dirs.output_dir / "results_1a_lmm.txt", "w") as f:
        f.write(output_str)

Running model 'lmm'
✅ Model 'lmm' estimated: peer_tfp_ttwa_donut=0.033, peer_tfp_pc4_donut=0.015, peer_tfp_pc8=0.280
Running model 'lmm_no_firm_fe'
✅ Model 'lmm_no_firm_fe' estimated: peer_tfp_ttwa_donut=0.204, peer_tfp_pc4_donut=0.142, peer_tfp_pc8=0.559
Running model 'lmm_employees'
✅ Model 'lmm_employees' estimated: peer_tfp_ttwa_donut=0.306, peer_tfp_pc4_donut=0.015, peer_tfp_pc8=0.280, ln_employees=0.184, peer_tfp_pc8_x_employees=-0.058
Panel regressions complete. 3 models, writing.


# 1b. Industries group model

# 2. Distance decay model

$$
\begin{align*}
z_{it} &= \alpha_i + \gamma_t + \rho \sum_{j \neq i} f(d_{ij}) \cdot z_{jt} + \epsilon_{it}   \\
y_{it} &= \alpha_i + \gamma_t + \beta_1 k_{it} + \beta_2 l_{it} + \rho \sum_{j \neq i} w_{ij} z_{jt} + \epsilon_{it}
\end{align*}
$$

In [59]:
table_panel_name = "working_yearly_with_tfp_wave"

# 3. Models: varying Y (gva1 vs gva2) and K (fixed_total vs tangibles)
models = {
    'dd_gravity1': {
        'Y': 'tfp',
        'X': ['tfp_wav1'],
        'W': ['nb_peers', 'employees'],
        'to_log': ['nb_peers', 'employees'],
        'fe': ['i', 't'],
        'description': 'Base: 1/d distance peer effect'
    },
    'dd_neg2': {
        'Y': 'tfp',
        'X': ['tfp_wav2'],
        'W': ['nb_peers', 'employees'],
        'to_log': ['nb_peers', 'employees'],
        'fe': ['i', 't'],
        'description': 'Base: -1/d^2 distance peer effect'
    },
    'dd_expdd3': {
        'Y': 'tfp',
        'X': ['tfp_wav3'],
        'W': ['nb_peers', 'employees'],
        'to_log': ['nb_peers', 'employees'],
        'fe': ['i', 't'],
        'description': 'Base: exp(-d / 1000) distance peer effect'
    },
    'dd_struct1': {
        'Y': 'gva1',
        'X': ['tfp_wav1'],
        'W': ['total_assets', 'employees'],
        'to_log': ['gva1', 'total_assets', 'employees'],
        'fe': ['i', 't'],
        'description': 'Structural GVA model regressing with peer TFP'
    },
    'dd_struct2': {
        'Y': 'gva1',
        'X': ['tfp_wav2'],
        'W': ['total_assets', 'employees'],
        'to_log': ['gva1', 'total_assets', 'employees'],
        'fe': ['i', 't'],
        'description': 'Structural GVA model regressing with peer TFP'
    },
    'dd_struct3': {
        'Y': 'gva1',
        'X': ['tfp_wav3'],
        'W': ['total_assets', 'employees'],
        'to_log': ['gva1', 'total_assets', 'employees'],
        'fe': ['i', 't'],
        'description': 'Structural GVA model regressing with peer TFP'
    },
}

run_res_series: list[tuple[PanelEffectsResults, ModelSpec, str]] = []
worker_args = [
    (ModelSpec(**mod_obj), df_panel, m_name)
    for m_name, mod_obj in models.items() if ModelSpec(**mod_obj).include
]
for args in worker_args:
    mod, table_panel_name, model_name = args
    res, beta, effects_dict = run_panel(args)
    if res is None:
        print(f"Model '{model_name}' failed. Skipping.")
        continue
    run_res_series.append((res, mod, model_name))

model_count = len(run_res_series)
if model_count:
    output_str = "\n".join(map(format_str, run_res_series))
    out_file = dirs.output_dir / "results_2_dd"
    print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}, writing to {out_file}")
    with open(f"{out_file}.txt", "w") as f:
        f.write(output_str)

Running model 'dd_gravity1'
✅ Model 'dd_gravity1' estimated: tfp_wav1=0.168, ln_nb_peers=0.013, ln_employees=-0.009
Running model 'dd_neg2'
✅ Model 'dd_neg2' estimated: tfp_wav2=0.135, ln_nb_peers=0.012, ln_employees=-0.009
Running model 'dd_expdd3'
✅ Model 'dd_expdd3' estimated: tfp_wav3=0.184, ln_nb_peers=0.016, ln_employees=-0.009
Running model 'dd_struct1'
✅ Model 'dd_struct1' estimated: tfp_wav1=0.168, ln_total_assets=0.265, ln_employees=0.619
Running model 'dd_struct2'
✅ Model 'dd_struct2' estimated: tfp_wav2=0.135, ln_total_assets=0.265, ln_employees=0.619
Running model 'dd_struct3'
✅ Model 'dd_struct3' estimated: tfp_wav3=0.183, ln_total_assets=0.265, ln_employees=0.619
Panel regressions complete. 6 models, writing to C:\Users\lazyst\Files\ucl\Dissertation\model\output\results_2_dd
